# Staff Allocation to Projects — Multi-objective MILP

Works on `toy_instance.json`, `medium_instance.json`, `large_instance.json` and on generated instances. Uses SciPy/HiGHS (`pip install numpy scipy pandas matplotlib`).

**What changed vs. the previous version**
- Model construction is vectorised (sparse COO) — previously quadratic loops/`lil_matrix`, far too slow for medium/large.
- Completion-time constraint uses a sparse auxiliary instead of a dense sum per assignment.
- All objectives are **recomputed from the actual assignments**, so reported values are always consistent with the schedule.
- Epsilon-constraint front: exact on the toy instance (all 576 integer bounds, infeasibility pruning); on medium/large a **budgeted grid** (full enumeration is intractable) — results are flagged as approximate.
- Objective-space and decision-space (Gantt) visualisations, as required by the brief, plus an instance generator.

## Mathematical formulation

Sets: employees $i\in I$, projects $j\in J$, qualifications $q\in Q$, days $t\in\{1,\ldots,H\}$.

Parameters: $r_{jq}$ = required person-days of skill $q$ for project $j$; $g_j$ = project gain; $d_j$ = due date; $p_j$ = daily late penalty; $a_{iq}=1$ if employee $i$ has qualification $q$; $v_{it}=1$ if employee $i$ is on leave on day $t$.

Binary decisions: $x_{ijqt}=1$ if employee $i$ works on project $j$ with skill $q$ on day $t$; $y_j=1$ if project $j$ is completed; $c_{jt}=1$ if project $j$ completes on day $t$; $u_{ij}=1$ if employee $i$ works on project $j$. Continuous $M$ is the makespan.

**Constraints**

1. Qualification eligibility: create assignment variables only for qualified employees.
2. One assignment per employee/day: $\sum_{j,q}x_{ijqt}\leq 1$.
3. Leave: create no assignment variable on an employee's vacation days.
4. Skill coverage: $\sum_{i,t}x_{ijqt}=r_{jq}y_j$ for every required project-skill pair.
5. Project executed at most once: $\sum_t c_{jt}=y_j$.
6. Assignments occur no later than completion: $x_{ijqt}\leq\sum_{\tau=t}^Hc_{j\tau}$.
7. Link employee-project indicator: $x_{ijqt}\leq u_{ij}$ and $u_{ij}\leq\sum_{q,t}x_{ijqt}$.
   (In the code, constraint 6 uses an auxiliary continuous $k_{jt}=\sum_{\tau\ge t}c_{j\tau}$ with $k_{jt}-k_{j,t+1}=c_{jt}$, which keeps the matrix sparse.)
8. Makespan: $M\geq t c_{jt}$.

**Objectives**

$f_1=\sum_jg_jy_j-\sum_{j,t>d_j}p_j(t-d_j)c_{jt}$ (maximize net profit).

$f_2=M$ (minimize makespan); $f_3=\sum_{i,j}u_{ij}$ (minimize employee-project relationships); $f_4=\sum_{j,t>d_j}c_{jt}$ (minimize late projects).

Assumption: required person-days can be performed in parallel by different qualified employees; tasks need not be consecutive and have no precedence constraints. A project's completion day is the latest day on which its work is performed.


In [ ]:
import json, time, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix, csr_matrix, vstack


In [ ]:
def build_model(data):
    staff, jobs, H = data['staff'], data['jobs'], data['horizon']
    I, J = len(staff), len(jobs)
    xkeys = []
    for i, p in enumerate(staff):
        quals, leave = set(p['qualifications']), set(p.get('vacations', []))
        for j, job in enumerate(jobs):
            for q in job['working_days_per_qualification']:
                if q in quals:
                    xkeys.extend((i, j, q, t) for t in range(1, H+1) if t not in leave)
    n = len(xkeys)
    xidx = {k: c for c, k in enumerate(xkeys)}
    yidx = {j: n + j for j in range(J)}
    cidx = {(j, t): n + J + j*H + t-1 for j in range(J) for t in range(1, H+1)}
    kbase = n + J + J*H                      # k_jt = 1 if project j still open on day t (completion >= t)
    kidx = {(j, t): kbase + j*H + t-1 for j in range(J) for t in range(1, H+1)}
    ubase = kbase + J*H
    uidx = {(i, j): ubase + i*J + j for i in range(I) for j in range(J)}
    midx = ubase + I*J
    nv = midx + 1
    R, C, V, lo, hi = [], [], [], [], []
    def row(entries, l=-np.inf, h=np.inf):
        r = len(lo)
        for c, v in entries: R.append(r); C.append(c); V.append(v)
        lo.append(l); hi.append(h)
    by_it, by_jq, by_ij = {}, {}, {}
    for k, c in xidx.items():
        i, j, q, t = k
        by_it.setdefault((i, t), []).append(c)
        by_jq.setdefault((j, q), []).append(c)
        by_ij.setdefault((i, j), []).append(c)
    for cols in by_it.values(): row([(c, 1) for c in cols], h=1)                 # one task / employee / day
    for j, job in enumerate(jobs):
        for q, req in job['working_days_per_qualification'].items():              # skill coverage
            row([(c, 1) for c in by_jq.get((j, q), [])] + [(yidx[j], -req)], 0, 0)
    for j in range(J):
        row([(cidx[(j, t)], 1) for t in range(1, H+1)] + [(yidx[j], -1)], 0, 0)   # one completion day
        for t in range(1, H+1):                                                   # k_jt - k_j,t+1 = c_jt
            e = [(kidx[(j, t)], 1), (cidx[(j, t)], -1)]
            if t < H: e.append((kidx[(j, t+1)], -1))
            row(e, 0, 0)
        for t in range(1, H+1): row([(cidx[(j, t)], t), (midx, -1)], h=0)         # makespan
    for (i, j, q, t), c in xidx.items():
        row([(c, 1), (kidx[(j, t)], -1)], h=0)                                    # work only before completion
        row([(c, 1), (uidx[(i, j)], -1)], h=0)                                    # x <= u
    for (i, j), u in uidx.items():
        row([(u, 1)] + [(c, -1) for c in by_ij.get((i, j), [])], h=0)             # u <= sum x
    A = coo_matrix((V, (R, C)), shape=(len(lo), nv)).tocsr()
    lb, ub = np.zeros(nv), np.ones(nv); ub[midx] = H
    integ = np.ones(nv, dtype=int); integ[midx] = 0
    for c in kidx.values(): integ[c] = 0
    return dict(data=data, xkeys=xkeys, xidx=xidx, yidx=yidx, cidx=cidx, kidx=kidx, uidx=uidx,
                midx=midx, nv=nv, A=A, clb=np.array(lo), cub=np.array(hi), lb=lb, ub=ub, integrality=integ)

def objective_rows(model):
    d = model['data']; jobs, H = d['jobs'], d['horizon']; n = model['nv']
    profit = np.zeros(n); late = np.zeros(n); div = np.zeros(n); mk = np.zeros(n)
    mk[model['midx']] = 1
    for j, job in enumerate(jobs):
        profit[model['yidx'][j]] = -job['gain']                # minimise -gross profit
        for t in range(job['due_date']+1, H+1):
            ci = model['cidx'][(j, t)]
            profit[ci] = job['daily_penalty']*(t-job['due_date']); late[ci] = 1
    for u in model['uidx'].values(): div[u] = 1
    return profit, mk, div, late

def extract(model, z):
    """Recompute all objective values from the actual assignments x (not from c)."""
    d = model['data']; jobs, staff, H = d['jobs'], d['staff'], d['horizon']
    sched = {}                                    # (employee, day) -> (project, skill)
    last = {}
    for (i, j, q, t), c in model['xidx'].items():
        if z[c] > .5:
            sched[(staff[i]['name'], t)] = (jobs[j]['name'], q); last[j] = max(last.get(j, 0), t)
    done = [j for j in range(len(jobs)) if z[model['yidx'][j]] > .5]
    pen = sum(jobs[j]['daily_penalty']*max(0, last[j]-jobs[j]['due_date']) for j in done)
    pairs = len({(e, p) for (e, t), (p, q) in sched.items()})
    return dict(net_profit=sum(jobs[j]['gain'] for j in done)-pen,
                makespan=max(last.values(), default=0), employee_project_pairs=pairs,
                late_projects=sum(last[j] > jobs[j]['due_date'] for j in done),
                completed_projects=[jobs[j]['name'] for j in done], schedule=sched)

KEYS = ('net_profit', 'makespan', 'employee_project_pairs', 'late_projects')
def vec(s): return (s['net_profit'], -s['makespan'], -s['employee_project_pairs'], -s['late_projects'])
def dominates(a, b):
    va, vb = vec(a), vec(b); return all(x >= y for x, y in zip(va, vb)) and va != vb
def pareto_filter(sols):
    u = {}
    for s in sols: u.setdefault(tuple(s[k] for k in KEYS), s)
    v = list(u.values())
    return sorted([s for s in v if not any(dominates(o, s) for o in v)], key=lambda s: (-s['net_profit'], s['makespan']))

def solve_weighted(model, weights, time_limit=30, gap=0.0, nonempty=True):
    d = model['data']; jobs, H = d['jobs'], d['horizon']; I, J = len(d['staff']), len(jobs)
    profit, mk, div, late = objective_rows(model)
    ps = max(1, sum(j['gain'] for j in jobs))
    obj = weights[0]*profit/ps + weights[1]*mk/H + weights[2]*div/max(1, I*J) + weights[3]*late/max(1, J)
    A, lo, hi = model['A'], model['clb'], model['cub']
    cons = LinearConstraint(A, lo, hi)
    r = milp(obj, integrality=model['integrality'], bounds=Bounds(model['lb'], model['ub']),
             constraints=cons, options=dict(time_limit=time_limit, mip_rel_gap=gap))
    if r.x is None: return None, r
    s = extract(model, r.x); s['weights'] = tuple(weights); s['proven_optimal'] = (r.status == 0)
    return s, r

def epsilon_front(model, m_grid=None, d_grid=None, l_grid=None, time_limit=20, budget=None, verbose=False):
    """Epsilon-constraint: maximise net profit s.t. makespan<=m, pairs<=d, late<=l.
    Full integer grids + proven optimality on every solve => exact Pareto front (certified=True).
    Coarser grids / time limits / budget => approximation (certified=False)."""
    d = model['data']; H, I, J = d['horizon'], len(d['staff']), len(d['jobs'])
    profit, mk, div, late = objective_rows(model)
    ps = max(1, sum(j['gain'] for j in d['jobs']))
    # tiny secondary weights: pushes returned solutions towards low m/d/l (fewer dominated points)
    obj = profit/ps + 1e-4*(mk/H + div/max(1, I*J) + late/max(1, J))
    extra = csr_matrix(np.stack([mk, div, late]))
    A = vstack([model['A'], extra], format='csr')
    lo = np.r_[model['clb'], [-np.inf]*3]
    m_grid = list(range(H+1)) if m_grid is None else sorted(set(m_grid))
    d_grid = list(range(I*J+1)) if d_grid is None else sorted(set(d_grid))
    l_grid = list(range(J+1)) if l_grid is None else sorted(set(l_grid))
    exact_grids = (len(m_grid) == H+1 and len(d_grid) == I*J+1 and len(l_grid) == J+1)
    cands, infeas, certified, n, t0 = [], [], exact_grids, 0, time.time()
    for l in l_grid:
        for m in m_grid:
            for dd in d_grid:
                if any(m <= a and dd <= b and l <= c for a, b, c in infeas): continue
                if budget and time.time()-t0 > budget: certified = False; return pareto_filter(cands), certified, n
                hi = np.r_[model['cub'], [m, dd, l]]
                n += 1
                r = milp(obj, integrality=model['integrality'], bounds=Bounds(model['lb'], model['ub']),
                         constraints=LinearConstraint(A, lo, hi), options=dict(time_limit=time_limit, mip_rel_gap=0.0))
                if r.status == 2: infeas.append((m, dd, l)); continue
                if r.x is None: certified = False; continue
                if r.status != 0: certified = False
                s = extract(model, r.x); cands.append(s)
                if verbose: print(n, (m, dd, l), {k: s[k] for k in KEYS})
    return pareto_filter(cands), certified, n

def instance_generator(n_staff=5, n_jobs=15, n_skills=8, horizon=24, seed=0, vac_frac=0.15):
    rnd = random.Random(seed)
    Q = [chr(65+k) for k in range(n_skills)]
    staff = []
    for k in range(n_staff):
        nq = rnd.randint(2, min(4, n_skills))
        nv = rnd.randint(0, max(1, int(vac_frac*horizon)))
        start = rnd.randint(1, max(1, horizon-nv)) if nv else 1
        staff.append(dict(name=f'Emp{k+1}', qualifications=sorted(rnd.sample(Q, nq)),
                          vacations=list(range(start, start+nv))))
    jobs = []
    for k in range(n_jobs):
        sk = rnd.sample(Q, rnd.randint(1, min(3, n_skills)))
        req = {q: rnd.randint(1, 8) for q in sk}
        tot = sum(req.values())
        jobs.append(dict(name=f'Job{k+1}', gain=int(5*tot*rnd.uniform(.8, 2)), due_date=rnd.randint(max(2, horizon//3), horizon),
                         daily_penalty=3, working_days_per_qualification=req))
    return dict(horizon=horizon, qualifications=Q, staff=staff, jobs=jobs)


In [ ]:
def plot_objective_space(front, title='', ax=None):
    import matplotlib.pyplot as plt, pandas as pd
    from pandas.plotting import parallel_coordinates
    df = pd.DataFrame([{k: s[k] for k in KEYS} for s in front])
    fig = plt.figure(figsize=(13, 4.5))
    a1 = fig.add_subplot(1, 3, 1); a1.scatter(df.makespan, df.net_profit, c=df.late_projects, cmap='viridis', s=60)
    a1.set_xlabel('makespan'); a1.set_ylabel('net profit'); a1.set_title('profit vs makespan (colour = #late)')
    a2 = fig.add_subplot(1, 3, 2); sc = a2.scatter(df.employee_project_pairs, df.net_profit, c=df.makespan, cmap='plasma', s=60)
    a2.set_xlabel('employee-project pairs'); a2.set_ylabel('net profit'); plt.colorbar(sc, ax=a2, label='makespan')
    a3 = fig.add_subplot(1, 3, 3)
    norm = (df-df.min())/(df.max()-df.min()).replace(0, 1)
    for _, r in norm.iterrows(): a3.plot(range(4), r.values, alpha=.5)
    a3.set_xticks(range(4)); a3.set_xticklabels(['profit', 'makespan', 'pairs', 'late'], fontsize=8); a3.set_title('parallel coordinates (normalised)')
    fig.suptitle(title); fig.tight_layout(); return fig

def plot_schedule(model, sol, title=''):
    import matplotlib.pyplot as plt
    d = model['data']; H = d['horizon']; names = [p['name'] for p in d['staff']]
    jn = [j['name'] for j in d['jobs']]; cmap = plt.get_cmap('tab20')
    fig, ax = plt.subplots(figsize=(min(18, 0.45*H+3), 0.6*len(names)+1.5))
    for r, e in enumerate(names):
        for t in range(1, H+1):
            if t in set(d['staff'][r].get('vacations', [])): ax.add_patch(plt.Rectangle((t-1, r), 1, 1, color='lightgrey', hatch='//'))
            elif (e, t) in sol['schedule']:
                p, q = sol['schedule'][(e, t)]; ax.add_patch(plt.Rectangle((t-1, r), 1, 1, color=cmap(jn.index(p) % 20), ec='white'))
                ax.text(t-.5, r+.5, f"{p.replace('Job', 'J')}\n{q}", ha='center', va='center', fontsize=6)
    ax.set_xlim(0, H); ax.set_ylim(0, len(names)); ax.invert_yaxis()
    ax.set_yticks([r+.5 for r in range(len(names))]); ax.set_yticklabels(names); ax.set_xticks(range(0, H+1, 2)); ax.set_xlabel('day (grey = leave)')
    ax.set_title(title or f"profit={sol['net_profit']} makespan={sol['makespan']} pairs={sol['employee_project_pairs']} late={sol['late_projects']}")
    fig.tight_layout(); return fig



In [ ]:
def auto_grids(model, k_m=5, k_d=5, l_vals=(0, 1, 2, 4)):
    d = model['data']; H, I, J = d['horizon'], len(d['staff']), len(d['jobs'])
    m = sorted(set(int(round(x)) for x in np.linspace(H//3, H, k_m)))
    dd = sorted(set(int(round(x)) for x in np.linspace(max(1, J//3), min(I*J, 2*J), k_d)))
    l = sorted(set([x for x in l_vals if x < J] + [J]))
    return m, dd, l


## Run: toy (exact), medium and large (budgeted)

`BUDGET` is seconds of solver time per instance for medium/large. Raise it (and `TIME_LIMIT`) for a denser front. A front is **certified exact only when `certified=True`**.

In [ ]:
TIME_LIMIT, BUDGET = 10, 300      # per-MILP limit and total budget (s) for medium/large
models, fronts = {}, {}
for name in ['toy', 'medium', 'large']:
    data = json.load(open(f'{name}_instance.json', encoding='utf-8'))
    models[name] = build_model(data)
    t = time.time()
    if name == 'toy':
        fr, cert, n = epsilon_front(models[name])                      # full integer grids -> exact
    else:
        fr, cert, n = epsilon_front(models[name], *auto_grids(models[name]), time_limit=TIME_LIMIT, budget=BUDGET)
    fronts[name] = fr
    print(f'{name}: {len(fr)} non-dominated solutions | {n} MILPs | certified exact: {cert} | {time.time()-t:.0f}s')
    display(pd.DataFrame([{k: s[k] for k in KEYS} | {'projects': len(s['completed_projects'])} for s in fr]))


## Objective space

In [ ]:
for name in fronts:
    plot_objective_space(fronts[name], title=f'{name} instance'); plt.show()


## Decision space (schedule of a chosen solution)

Change `pick` to inspect another point of the front (index in the table above).

In [ ]:
pick = 0
for name in fronts:
    plot_schedule(models[name], fronts[name][min(pick, len(fronts[name])-1)], title=f'{name} — solution {pick}'); plt.show()


## Weighted-sum runs (optional, for comparison)

Samples only *supported* Pareto points.

In [ ]:
from itertools import product
weights = [tuple(v/5 for v in w) for w in product(range(6), repeat=4) if sum(w) == 5]
m = models['toy']
cands = [s for w in weights if (s := solve_weighted(m, w, time_limit=10)[0])]
print(len(weights), 'weight vectors ->', len(pareto_filter(cands)), 'non-dominated (exact front has', len(fronts['toy']), ')')


## Instance generator

Brief requires a generator to test scalability.

In [ ]:
gen = instance_generator(n_staff=6, n_jobs=20, n_skills=8, horizon=30, seed=1)
gm = build_model(gen)
fr, cert, n = epsilon_front(gm, *auto_grids(gm), time_limit=5, budget=120)
print(len(fr), 'solutions, certified:', cert)
plot_objective_space(fr, 'generated'); plt.show(); plot_schedule(gm, fr[0]); plt.show()


Note: the empty schedule (all objectives 0) is a valid non-dominated point since the brief does not force selecting a project; filter it out if undesired.